# 7 DuckDB SQL Queries That Save You Hours of Pandas Work

This article is available at: https://www.kdnuggets.com/7-duckdb-sql-queries-that-save-you-hours-of-pandas-work

In this workbook we consider the questions (from: https://platform.stratascratch.com/data-projects/partner-business-modeling)
Note: It appears this assignment is given to potential employees at Uber.

Assignment

**Scenario 1**: 
It is going to be a huge Saturday and there will need to be many more cars on the road than last week. In order to get drivers to go online, we're assessing the following two bonus options in terms of cost:

- Option 1: $50 for each driver that is online at least 8 hours, accepts 90% of requests, completes 10 trips, and has a rating of 4.7 or better during the time frame;
  
- Option 2: $4/trip for all drivers who complete 12 trips, and have a 4.7 or better rating.
  
Using the dataset provided and given Scenario 1, provide answers to the questions below:
1. How much would the total bonus payout be with Option 1?
2. How much would the total bonus payout be with Option 2?
3. How many drivers would qualify for a bonus under Option 1 but not under Option 2?
4. What percentages of drivers online completed less than 10 trips, had an acceptance rate of less than 90%, and had a rating of 4.7 or higher?

**Scenario 2**: A taxi driver currently generates $200 per day in fares (before expenses), works six days a week, takes three weeks off, and has the following expenses:

- Gas - $200 per week
  
- Insurance - $400 per month

  
- Vehicle rent (by the week) - $500

The driver doesn't pay gas and rent expenses on off weeks.
Now, let's assume that the same driver would buy a Town Car and partner with Uber. If he does, his gas expenses would go up by 5%, his insurance expense would decrease by 20%, and he would no longer be renting a vehicle. However, he would need to buy a car. The driver would still take three weeks off per year.

Given Scenario 2, provide answers to the questions below:

5. How much money (after expenses) does the taxi driver make per year without partnering with Uber?
6. You are convincing the same driver above to buy a Town Car and partner with Uber. Assuming the new car is 40,000 USD, how much would the driver's gross fares need to increase per week to fully pay for the car in year 1 and maintain the same yearly profit margin as before?


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('dataset_2.csv')

In [3]:
df.head()

,Name,Trips Completed,Accept Rate,Supply Hours,Rating
0,Abdul,1,100%,3,4.8
1,Abraham,12,83%,5,4.7
2,Adelina,1,100%,2,4.7
3,Akilah,1,100%,2,4.9
4,Alec,21,76%,11,5.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             119 non-null    object 
 1   Trips Completed  119 non-null    int64  
 2   Accept Rate      119 non-null    object 
 3   Supply Hours     119 non-null    int64  
 4   Rating           119 non-null    float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.8+ KB


In [5]:
import duckdb

In [6]:
con = duckdb.connect()

In [7]:
con.register("my_data", df)

Find drivers that
- are online for at least 8 hours
- complete at least 10 trips
- accept at least 90% of ride requests
- have a rating of 4.7 or above

In [10]:
con.execute("""
SELECT 
    COUNT(*) AS qualified_drivers,
    COUNT(*) * 50 AS total_payout
FROM my_data
WHERE "Supply Hours" >= 8
    AND CAST(REPLACE("Accept Rate", '%', '') AS DOUBLE) >= 90
    AND "Trips Completed" >= 10
    AND Rating >= 4.7
""").fetchdf()

,qualified_drivers,total_payout
0,21,1050


Option 2: Drivers will receive $4 per trip if:
- They complete at least 12 trips
- Have a rating of 4.7 or better

In [11]:
con.execute("""
SELECT 
    COUNT(*) AS qualified_drivers,
    SUM("Trips Completed") * 4 AS total_payout
FROM my_data
WHERE "Trips Completed" >= 12
    AND Rating >= 4.7
""").fetchdf()

,qualified_drivers,total_payout
0,46,2976.0


3. How many drivers would qualify for a bonus under Option 1 but not under Option 2?

In [12]:
con.execute("""
SELECT COUNT(*) AS only_option1
FROM my_data
WHERE "Supply Hours" >= 8
    AND CAST(REPLACE("Accept Rate", '%', '') AS DOUBLE) >= 90
    AND "Trips Completed" >= 10
    AND Rating >= 4.7
    AND NOT ("Trips Completed" >= 12 AND Rating >= 4.7)
""").fetchdf()

,only_option1
0,2


4. What percentages of drivers online completed less than 10 trips, had an acceptance rate of less than 90%, and had a rating of 4.7 or higher?

In [19]:
con.execute("""
SELECT
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM my_data), 2) AS percentage
FROM my_data
WHERE "Trips Completed" < 10
    AND CAST(REPLACE("Accept Rate", '%', '') AS DOUBLE) < 90
    AND "Rating" >= 4.7
""").fetchdf()

,percentage
0,10.92


In [16]:
con.execute("""
SELECT COUNT(*) FROM my_data
""").fetchdf()

,count_star()
0,119


In [20]:
con.execute("""
SELECT COUNT(*)
FROM my_data
WHERE "Trips Completed" < 10
    AND CAST(REPLACE("Accept Rate", '%', '') AS DOUBLE) < 90
    AND "Rating" >= 4.7
""").fetchdf()

,count_star()
0,13


In [21]:
con.execute("""
SELECT 
    200 * 6 * (52 - 3) AS annual_revenue,
    200 * (52 - 3) AS gas_expense,
    500 * (52 - 3) AS rent_expense,
    400 * 12 AS insurance_expense,
    (200 * 6 * (52 - 3)) 
      - (200 * (52 - 3) + 500 * (52 - 3) + 400 * 12) AS net_income
""").fetchdf()

,annual_revenue,gas_expense,rent_expense,insurance_expense,net_income
0,58800,9800,24500,4800,19700
